## 1. Load expert and amateur models

### Optional: Uninstall the Dataset

If the dataset has already been loaded, you may need to uninstall it to avoid conflicts.  
**Remove the `#` symbol below to run the command.**

```python
!pip uninstall datasets
```

In [ ]:
!pip install transformers
# !pip uninstall datasets
!pip install datasets

Found existing installation: datasets 2.14.4
Uninstalling datasets-2.14.4:
  Would remove:
    /usr/local/bin/datasets-cli
    /usr/local/lib/python3.11/dist-packages/datasets-2.14.4.dist-info/*
    /usr/local/lib/python3.11/dist-packages/datasets/*
Proceed (Y/n)? y
  Successfully uninstalled datasets-2.14.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_m

## Login HuggingFace


In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `henrytran0715` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `henr

In [ ]:
!pip install --upgrade transformers

or using this on runpod

In [ ]:
from huggingface_hub import login

login(token="use-your-token")  # Replace with your actual token

## General Data Loading Setup

In [ ]:
import random
import torch
import numpy as np
from datasets import load_dataset
from torch.utils.data import DataLoader

# Set seeds for reproducibility
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

def generate_dataset(dataset_name, if_gsm8k = False):
    # Load dataset splits
    if if_gsm8k:
      train_dataset = load_dataset(dataset_name, 'main', split="train")
      test_dataset = load_dataset(dataset_name, 'main', split="test")
    else:
      train_dataset = load_dataset(dataset_name, split="train")
      test_dataset = load_dataset(dataset_name, split="test")

    # Generate 5 random indices from the training set
    rand_indices = [random.randint(0, len(train_dataset) - 1) for _ in range(5)]
    debug_dataset = train_dataset.select(rand_indices)

    return train_dataset, test_dataset, debug_dataset

In [ ]:
## For training purposes, we should increase the batch size to values like 16, 32, or 64 — typically powers of 2. For now, batch size = 1 for debug

batch_size = 1
def generate_dataloader(train_dataset, test_dataset, debug_dataset):
  train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle = False)
  test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle = False)
  debug_dataloader = DataLoader(debug_dataset, batch_size=batch_size, shuffle = False)

  return (train_dataloader, test_dataloader, debug_dataloader)

In [ ]:
def print_first_features_from_dataset(data, feature1, feature2):
    for i, item in enumerate(data):
        if i == 0:
            first_feature = data[feature1][0]
            second_feature = data[feature2][0]
            print(first_feature)
            print(second_feature)

In [ ]:
def gsm8k_clean_data(train_dataset, test_dataset, debug_dataset):
    import re

    def clean_data(text):
        return re.sub(r'<<.*?>>', '', text)

    train_dataset = train_dataset.map(lambda x: {"answer": clean_data(x["answer"])})
    test_dataset = test_dataset.map(lambda x: {"answer": clean_data(x["answer"])})
    debug_dataset = debug_dataset.map(lambda x: {"answer": clean_data(x["answer"])})

    return train_dataset, test_dataset, debug_dataset

### Loading Data and Models

This "Save Processed Data" section only needs to be run once.
It sets up the expert, amateur, and base models along with the relevant datasets,
and saves them to Google Drive for future use.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dataset_list = ["gsm8k", "commen_gen", "whatsthatbook"]
dataset_id = ["openai/gsm8k", "allenai/common_gen", "nlpkevinl/whatsthatbook"]

# Add expert into the list later
models_list = ["amateur", "base"]
checkpoint_list = ["meta-llama/Llama-2-7b-hf", "microsoft/phi-1_5"]

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

def loading_dataset(dataset_list, dataset_id_list):
    """
    Loop over dataset_list and save their train/test/debug splits to Google Drive.
    Returns:
        A list of tuples, each containing (train_dataset, test_dataset, debug_dataset) for each dataset.
    """
    all_datasets = []

    for i in range(len(dataset_list)):
        dataset_name = dataset_list[i]
        dataset_id = dataset_id_list[i]
        is_gsm8k = dataset_name.lower() == "gsm8k"

        # Generate datasets
        train_dataset, test_dataset, debug_dataset = generate_dataset(dataset_id, is_gsm8k)

        if is_gsm8k:
            train_dataset, test_dataset, debug_dataset = gsm8k_clean_data(train_dataset, test_dataset, debug_dataset)
        # Generate dataloaders
        train_dataloader, test_dataloader, debug_dataloader = generate_dataloader(
            train_dataset, test_dataset, debug_dataset
        )

        # Optionally collect datasets for further use
        all_datasets.append((train_dataset, test_dataset, debug_dataset))

    return all_datasets

def loading_model(model_name_list, checkpoint_path_list):
    """
    Load models and tokenizers from checkpoints and save them under model_name_list.
    """

    models_and_tokenizers = []

    for i in range(len(model_name_list)):
        checkpoint_path = checkpoint_path_list[i]
        model_name = model_name_list[i]

        tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
        model = AutoModelForCausalLM.from_pretrained(checkpoint_path)

        models_and_tokenizers.append({
          "model_name" : model_name_list[i],
          "model" : model,
          "tokenizer" : tokenizer
        })

    return models_and_tokenizers

### Save/Reload Data

In [ ]:
import csv
import os
import json

def save_results_to_csv(result, features, file_path):
    with open(file_path, 'a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        row = [result.get(feature, "") for feature in features]
        writer.writerow(row)


def converting_from_csv_to_json(features, csv_path, json_path):
    results = []
    with open(csv_path, 'r', encoding='utf-8') as file:
        csv_reader = csv.reader(file)

        for row in csv_reader:

            result = {feature: value for feature, value in zip(features, row)}
            results.append(result)

    with open(json_path, 'w', encoding='utf-8') as json_file:
        json.dump(results, json_file, indent=4, ensure_ascii=False)


In [ ]:
def reloading_result(file_path):
    with open(file_path, 'r') as file:
      results = json.load(file)

    return results

In [ ]:
datasets = loading_dataset(dataset_list, dataset_id)
models_and_tokenizers = loading_model(models_list, checkpoint_list)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

(…)d_posts_train_11552.split_with_val.jsonl:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

(…)gold_posts_dev_1444.split_with_val.jsonl:   0%|          | 0.00/2.70M [00:00<?, ?B/s]

(…)old_posts_test_1445.split_with_val.jsonl:   0%|          | 0.00/2.66M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11552 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15885 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1445 [00:00<?, ? examples/s]

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-2-7b-hf.
403 Client Error. (Request ID: Root=1-686577b9-4dd5397752e634c61553f22c;0fc6d49c-816f-4666-bab5-6cce5d9c052c)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-2-7b-hf/resolve/main/config.json.
Access to model meta-llama/Llama-2-7b-hf is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Llama-2-7b-hf to ask for access.

In [ ]:
# amateur model
amateur = models_and_tokenizers[0]["model"]
amateur_tokenizer = models_and_tokenizers[0]["tokenizer"]

In [ ]:
# base model
base = models_and_tokenizers[1]["model"]
base_tokenizer = models_and_tokenizers[1]["tokenizer"]

In [ ]:
# unpack gsm8k dataset

gsm8k_train_dataloader, gsm8k_test_dataloader, gsm8k_debug_dataloader = datasets[0]

# unpack common gen dataset

common_gen_train_dataloader, common_gen_test_dataloader, common_gen_debug_dataloader = datasets[1]

# unpack what's that book dataset

wtb_train_dataloader, wtb_test_dataloader, wtb_debug_dataloader = datasets[2]


# Check if gsm8k is loaded properly
print(gsm8k_train_dataloader.features)
print_first_features_from_dataset(gsm8k_train_dataloader, 'question', 'answer')

# Check if common_gen is loaded properly
print(common_gen_train_dataloader.features)
# print_first_features_from_dataset(common_gen_train_dataloader, 'question', 'answer')

# Check if gsm8k is loaded properly
print(wtb_train_dataloader.features)
# print_first_features_from_dataset(wtb_train_dataloader, 'question', 'answer')

In [ ]:
import torch

def set_device():
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"Using device: {device}")

  ## uncomment when having enough gpu
  # base_model.to(device)
  # amateur_model.to(device)
  # expert_model.to(device)
  return device

## GPT API feedback model designed for high-quality data generation.

In [ ]:
!pip install openai

In [ ]:
import openai
openai.api_key = "sk-xxxxxxxx"

In [ ]:
import json
import time
import re

class GPT4oFeedbackEvaluator:
    def __init__(self, model_name="gpt-4o-mini"):
        self.model_name = model_name
        self.expert_datasets = []


    def predict_score_and_feedback(self, prompt_text, answer):
        system_message = {
            "role": "system",
            "content": (
                "You are an expert evaluator. Your task is to score the quality of the following answer based "
                "on how well it responds to the given prompt.\n\n"
                "Use the following detailed scoring scale:\n"
                "1.0: Perfect completion with no issues.\n"
                "0.5 – 0.9: Generally good, minor flaws.\n"
                "0.1 – 0.4: Partially helpful, moderate issues.\n"
                "0.0: Neutral or off-topic, neither helpful nor harmful.\n"
                "-0.1 – -0.4: Mildly harmful or misleading.\n"
                "-0.5 – -0.9: Significantly off-task or misleading.\n"
                "-1.0: Complete failure or harmful content.\n\n"
                "Provide your reasoning and feedback in no more than 50 words, focusing on what was done well and what was missed.\n"
                "Do not comment on the length or verbosity of the answer.\n\n"
                "Respond exactly in this format:\n"
                "[reason]xxxx (MAX 50 words). Score: [score]\n\n"
                f"Prompt: {prompt}\n"
                f"Answer: {answer}\n"
            )
        }



        user_message = {
            "role": "user",
            "content": f"Prompt: {prompt_text}\nAnswer: {answer}\n"
        }

        try:
            response = openai.chat.completions.create(
                model=self.model_name,
                messages=[system_message, user_message],
                temperature=0,
                max_tokens=150,
                n=1
            )

            text = response.choices[0].message.content.strip()
            # print(f"Raw output:\n{text}")

            # Extract score
            score_match = re.search(r"Score\s*:\s*(-?\d+\.?\d*)", text, re.IGNORECASE)
            score = float(score_match.group(1)) if score_match else 0.0

            # Extract coverage and reasoning feedback (entire line before Score)
            feedback_line = text.split("Score")[0].strip()
            feedback = feedback_line

            # Store in expert_datasets
            self.expert_datasets.append({
                "prompt": prompt_text,
                "answer": answer,
                "score": score,
                "feedback": feedback
            })

        except Exception as e:
            score = 0.0
            feedback = f"Failed to parse score or feedback. Error: {e}"

        return score, feedback


    def generate_improved_answer(self, prompt, original_answer, feedback):
        system_message = {
            "role": "system",
            "content": (
                "You are an expert assistant. Your task is to read the prompt, the given answer, "
                "and the feedback, then generate an improved, accurate, and well-written answer that "
                "addresses the prompt fully and follows the feedback."
            )
        }
        user_message = {
            "role": "user",
            "content": (
                f"Prompt: {prompt}\n"
                f"Original Answer: {original_answer}\n"
                f"Feedback: {feedback}\n\n"
                "Please provide an improved answer."
            )
        }
        response = openai.chat.completions.create(
            model=self.model_name,
            messages=[system_message, user_message],
            temperature=0.7,
            max_tokens=300,
        )
        improved_answer = response.choices[0].message.content.strip()
        return improved_answer



In [ ]:
gpt_model = GPT4oFeedbackEvaluator()

In [ ]:
score, feedback = gpt_model.predict_score_and_feedback(
    prompt_text="Why do you like machine learning?",
    answer=(
        "Machine learning fascinates me"
    )
)

In [ ]:
print(feedback)
print(score)

[The response is vague and lacks depth, providing no specific reasons or insights into why machine learning is appealing.]
-0.19


## Setting up Expert Model

### 1k Sample Data Generation

In [ ]:
import random
import re

def corrupt_answer(answer):
    answer = answer.strip()
    if random.random() < 0.3:
        return random.choice([
            "I'm not sure about that.",
            "It might be something else.",
            "Maybe it's related to that topic.",
            "I don't have a clear answer.",
            "It depends on the context."
        ])
    if random.random() < 0.3:
        return " ".join(answer.split()[:2]) + " ..."

    swapped = re.sub(r"\b(\d{4})\b", lambda m: str(int(m.group(1)) + random.choice([-10, 10, 100])), answer)
    if swapped != answer:
        return swapped
    return answer + ", I think."

In [ ]:
from datasets import load_dataset, DatasetDict

dataset = load_dataset("sentence-transformers/natural-questions")
train_nq = dataset["train"]

In [ ]:
train_nq = train_nq.shuffle(seed=42).select(range(1500))

train_set = train_nq.select(range(1000))
test_set = train_nq.select(range(1200, 1500))

In [ ]:
import pprint

processed_datasets = []

for item in train_nq:
    prompt = item["query"]
    answer = item["answer"]

    if random.random() < 0.4:
        original_answer = corrupt_answer(answer)
        score, feedback = gpt_model.predict_score_and_feedback(prompt, original_answer)
        improved_answer = gpt_model.generate_improved_answer(prompt, original_answer, feedback)
    else:
        original_answer = answer
        score, feedback = gpt_model.predict_score_and_feedback(prompt, original_answer)
        improved_answer = gpt_model.generate_improved_answer(prompt, original_answer, feedback)

    example = {
        "prompt": prompt,
        "original_answer": original_answer,
        "feedback": feedback,
        "score" : score,
        "improved_answer": improved_answer
    }

    processed_datasets.append(example)

    pprint.pprint(example)

In [ ]:
len(processed_datasets)

Save the data to json file

In [ ]:
import json

output_path = "processed_feedback_dataset.jsonl"

with open(output_path, "w") as f:
    for example in processed_datasets:
        json_line = json.dumps(example)
        f.write(json_line + "\n")

print(f"Saved dataset to {output_path}")

Extract the data from the json file

In [ ]:
import json

file_path = "processed_feedback_dataset.jsonl"

dataset = []
with open(file_path, "r") as f:
    for line in f:
        example = json.loads(line)
        dataset.append(example)

print(dataset[0]["prompt"])
print(dataset[0]["original_answer"])
print(dataset[0]["feedback"])
print(dataset[0]["improved_answer"])

### Sample Dataset Entry

**Prompt:**  
`who are the basques and where do they live`

**Original Answer:**  
> The Basques (/bɑːsks/ or /bæsks/; Basque: euskaldunak; Spanish: vascos; French: basques) are an indigenous ethnic group characterized by the Basque language, a common culture, and shared ancestry to the ancient Vascones and Aquitanians.  
> They primarily inhabit the Basque Country (Euskal Herria), a region around the western end of the Pyrenees, straddling parts of north-central Spain and southwestern France.

**Score:** `1.0`

**Feedback:**  
> The answer is accurate, complete, and well-written. It provides a detailed explanation of who the Basques are...  

**Improved Answer:**  
> The Basques are an indigenous ethnic group primarily living in the Basque Country... They are characterized by their unique language, Basque, and a shared ancestry tracing back to the ancient Vascones and Aquitanians.


### Expert Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW
import re

class ExpertClass:
    def __init__(self, model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0", model_path=None, learning_rate=1e-5):
        """
        If model_path is provided, load model and tokenizer from there (local checkpoint).
        Otherwise, load from model_name (Hugging Face Hub).
        """
        if model_path:
            print(f"Loading model and tokenizer from local checkpoint: {model_path}")
            self.tokenizer = AutoTokenizer.from_pretrained(model_path)
            self.model = AutoModelForCausalLM.from_pretrained(model_path)
        else:
            print(f"Loading model and tokenizer from pretrained model name: {model_name}")
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForCausalLM.from_pretrained(model_name)

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        self.optimizer = AdamW(self.model.parameters(), lr=learning_rate)

    def prepare_inputs_and_labels(self, prompt, answer, score, feedback):
        full_text = (
             "You are an expert evaluator. Your task is to score the quality of the following answer based on how well it responds to the given prompt.\n"
            "Evaluate the answer on a scale from -1 to 1, where:\n"
            "-1 = completely incorrect, irrelevant, or does not address the prompt\n"
            "0 = neutral or partially addresses the prompt\n"
            "1 = excellent, fully correct, detailed, and thoroughly addresses the prompt\n\n"
            f"Prompt: {prompt}\n"
            f"Answer: {answer}\n\n"
            "Respond exactly in this format:\n"
            "Score: <score between -1 and 1>\n"
            "Feedback: <short, constructive feedback on how to improve the answer>\n"
        )

        tokenized = self.tokenizer(full_text, return_tensors="pt", truncation=True, max_length=1024)
        input_ids = tokenized.input_ids[0].clone()
        labels = input_ids.clone()

        score_token_ids = self.tokenizer("Score:", add_special_tokens=False).input_ids
        start_idx = None
        for i in range(len(input_ids) - len(score_token_ids) + 1):
            if torch.equal(input_ids[i:i+len(score_token_ids)], torch.tensor(score_token_ids)):
                start_idx = i
                break

        if start_idx is None:
            start_idx = 0

        labels[:start_idx] = -100

        return input_ids.unsqueeze(0).to(self.device), labels.unsqueeze(0).to(self.device), tokenized.attention_mask.to(self.device)

    ## This one is the problem. Should have better feedback score/prompt
    ##FROM ANDREW: You can use a prompt like this:
    #1.0: Completely fulfills the task with no noticeable issues.
    #0.5 to 0.9: Generally fulfills the task well, with minor issues or omissions.
    #0.1 to 0.4: Partially addresses the task but has moderate problems.
    #0.0: Neither helpful nor harmful — just neutral or off-topic.
    #-0.1 to -0.4: Actively unhelpful or misleading in minor ways.
    #-0.5 to -0.9: Significantly harmful, misleading, or off-task.
    #-1.0: Completely fails the task or provides harmful, misleading, or nonsensical content.
    def generate_feedback(self, prompt, answer, max_new_tokens=150):
        self.model.eval()

        input_text = (
            "You are an expert evaluator. Your task is to score the quality of the following answer based "
            "on how well it responds to the given prompt.\n\n"
            "Use the following detailed scoring scale:\n"
            "1.0: Perfect completion with no issues.\n"
            "0.5 – 0.9: Generally good, minor flaws.\n"
            "0.1 – 0.4: Partially helpful, moderate issues.\n"
            "0.0: Neutral or off-topic, neither helpful nor harmful.\n"
            "-0.1 – -0.4: Mildly harmful or misleading.\n"
            "-0.5 – -0.9: Significantly off-task or misleading.\n"
            "-1.0: Complete failure or harmful content.\n\n"
            "Provide your reasoning and feedback in no more than 50 words, focusing on what was done well and what was missed.\n"
            "Do not comment on the length or verbosity of the answer.\n\n"
            "Respond exactly in this format:\n"
            "[reason]xxxx (MAX 50 words). Score: [score]\n\n"
            f"Prompt: {prompt}\n"
            f"Answer: {answer}\n"
        )


        inputs = self.tokenizer(input_text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=0.7,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        generated_text = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)

        pattern = r"^\[(.*?)\].*Score:\s*(-?\d*\.?\d+)$"

        match = re.search(pattern, generated_text, re.DOTALL | re.MULTILINE)
        if match:
            feedback = match.group(1).strip()
            score = match.group(2).strip()
            return f"Score: {score}\nFeedback: {feedback}"
        else:
            return generated_text


    def fine_tuning(self, train_dataset, epochs=3, print_every=100):
        self.model.train()

        for epoch in range(epochs):
            total_loss = 0

            for i, sample in enumerate(train_dataset):
                prompt = sample.get("prompt", "")
                answer = sample.get("original_answer", "")
                score = sample.get("score", "")
                feedback = sample.get("feedback", "")

                input_ids, labels, attention_mask = self.prepare_inputs_and_labels(prompt, answer, score, feedback)

                outputs = self.model(input_ids=input_ids, labels=labels, attention_mask=attention_mask)
                loss = outputs.loss

                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()

                total_loss += loss.item()

                if (i + 1) % print_every == 0:
                    gen_output = self.generate_feedback(prompt, answer)

                    print(f"\n--- Sample {i+1} (Epoch {epoch+1}) ---")
                    print(f"Prompt: {prompt}")
                    print(f"Answer: {answer}")
                    print(f"Ground Truth Score: {score}")
                    print(f"Ground Truth Feedback:\n{feedback}")
                    print(f"\nGenerated Feedback:\n{gen_output}")
                    print(f"Loss: {loss.item():.4f}")
                    print("-" * 60)

            avg_loss = total_loss / len(train_dataset)
            print(f"\nEpoch {epoch+1} completed - Average Loss: {avg_loss:.4f}\n")

In [ ]:
expert = ExpertClass()

Loading model and tokenizer from pretrained model name: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [ ]:
expert.fine_tuning(train_set)

### [**Fine-tuned expert feedback model**](https://huggingface.co/henrytran07/expert_feedback_model/tree/main)

## Setting up Base Model

In [ ]:
prompt = "What are the benefits of being a machine learning engineer?"

In [ ]:
cot = """Q: California is nice.
A: California is renowned for its diverse geography and pleasant climate. From the sun-drenched beaches to the towering redwoods, the state offers incredible natural beauty. Its vibrant cities like Los Angeles and San Francisco attract millions with their culture and opportunities.

Q: {replace with the prompt}"""

### Processing the data before fine-tuning

In [ ]:
import json

file_path = "1ktrain_dataset.jsonl"
train_data = []

with open(file_path, "r") as f:
    for line in f:
        item = json.loads(line)

        prompt = item["prompt"]
        original_answer = item["original_answer"]
        improved_answer = item["improved_answer"]
        feedback = item["feedback"]
        train_data.append((prompt, original_answer, feedback, improved_answer))

print(f" {len(train_data)} examples loaded")

 1000 examples loaded


In [ ]:
from torch.utils.data import Dataset, DataLoader

class FeedbackDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    labels = [item["labels"] for item in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)
    attention_mask = (input_ids != tokenizer.pad_token_id).long()

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
import torch
from torch.optim import Adam
import openai
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

class BaseModel_Test:
    def __init__(self, model, tokenizer, feedback_model, penalty_factor=1.0):
        self.model = model.to("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = tokenizer
        self.feedback_model = feedback_model
        self.penalty_factor = penalty_factor
        self.device = self.model.device
        self.optimizer = Adam(self.model.parameters(), lr=3e-5)

        # Set pad and eos tokens if missing
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        if self.tokenizer.eos_token_id is None:
            self.tokenizer.eos_token = self.tokenizer.pad_token

        for param in self.model.parameters():
            param.requires_grad = True
        self.model.train()

    def format_sample(self, question, previous_answer, feedback, improved_answer):
        prompt = f"Q: {question}\n"
        if previous_answer:
            prompt += f"Previous Answer: {previous_answer}\n"
        if feedback:
            prompt += f"Feedback: {feedback}\n"
        prompt += "Based on the feedback above, please provide a revised and improved answer:\nA: "

        full_text = prompt + improved_answer

        return full_text

    def tokenize_sample(self, sample):
        question, previous_answer, feedback, improved_answer = sample

        full_text = self.format_sample(question, previous_answer, feedback, improved_answer)

        encodings = self.tokenizer(full_text, return_tensors="pt", truncation=True, max_length=512)

        input_ids = encodings.input_ids.squeeze(0)

        prompt = f"Q: {question}\n"
        if previous_answer:
            prompt += f"Previous Answer: {previous_answer}\n"
        if feedback:
            prompt += f"Feedback: {feedback}\n"
        prompt += "Based on the feedback above, please provide a revised and improved answer:\nA: "

        prompt_len = len(self.tokenizer(prompt, truncation=True, max_length=512).input_ids)
        labels = input_ids.clone()
        labels[:prompt_len] = -100

        return {
            "input_ids": input_ids,
            "labels": labels
        }


    def collate_fn(self, batch):
        input_ids = [item["input_ids"] for item in batch]
        labels = [item["labels"] for item in batch]

        input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)

        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

    def data_prep(self):
        train_split, val_split = train_test_split(train_data, test_size=0.2, random_state=42)

        tokenized_train = [self.tokenize_sample(x) for x in train_split]
        tokenized_val = [self.tokenize_sample(x) for x in val_split]

        train_loader = DataLoader(FeedbackDataset(tokenized_train), batch_size=4, shuffle=True, collate_fn=self.collate_fn)
        val_loader = DataLoader(FeedbackDataset(tokenized_val), batch_size=4, shuffle=False, collate_fn=self.collate_fn)

        return train_loader, val_loader

    def fine_tuning(self):
        train_loader, val_loader = self.data_prep()

        device = self.model.device
        self.model.train()

        for epoch in range(3):
            total_loss = 0
            for i, batch in enumerate(train_loader):
                batch = {k: v.to(device) for k, v in batch.items()}

                outputs = self.model(**batch)
                loss = outputs.loss

                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                total_loss += loss.item()

                if i % 10 == 0:
                    with torch.no_grad():
                        input_ids = batch["input_ids"]
                        attention_mask = batch["attention_mask"]

                        generated_ids = self.model.generate(
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                            max_new_tokens=200,
                            temperature=0.7,
                            top_p=0.9,
                            top_k=50,
                            repetition_penalty=1.2,
                            no_repeat_ngram_size=2,
                            do_sample=True,
                            pad_token_id=self.tokenizer.pad_token_id,
                            eos_token_id=self.tokenizer.eos_token_id,
                        )

                        pred_text = self.tokenizer.decode(generated_ids[0], skip_special_tokens=True)

                        labels = batch["labels"][0].clone()
                        labels[labels == -100] = self.tokenizer.pad_token_id
                        label_text = self.tokenizer.decode(labels, skip_special_tokens=True)

                        print(f"\nBatch {i} Prediction: {pred_text}")
                        print(f"Batch {i} Label: {label_text}\n")

            avg_loss = total_loss / len(train_loader)
            print(f"Epoch {epoch+1} | Average Loss: {avg_loss:.4f}")


            self.model.eval()
            val_loss = 0
            with torch.no_grad():
                for i, val_batch in enumerate(val_loader):
                    val_batch = {k: v.to(device) for k, v in val_batch.items()}

                    input_ids = val_batch["input_ids"]
                    attention_mask = val_batch["attention_mask"]

                    generated_ids = self.model.generate(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        max_new_tokens=200,
                        temperature=0.7,
                        top_p=0.9,
                        top_k=50,
                        repetition_penalty=1.2,
                        no_repeat_ngram_size=2,
                        do_sample=True,
                        pad_token_id=self.tokenizer.pad_token_id,
                        eos_token_id=self.tokenizer.eos_token_id,
                    )

                    pred_text = self.tokenizer.decode(generated_ids[0], skip_special_tokens=True)

                    labels = val_batch["labels"][0].clone()
                    labels[labels == -100] = self.tokenizer.pad_token_id
                    label_text = self.tokenizer.decode(labels, skip_special_tokens=True)

                    print(f"\nValidation Batch {i} Prediction: {pred_text}")
                    print(f"Validation Batch {i} Label: {label_text}\n")

                    outputs = self.model(**val_batch)
                    val_loss += outputs.loss.item()

            avg_val_loss = val_loss / len(val_loader)
            print(f"Epoch {epoch+1} | Average Validation Loss: {avg_val_loss:.4f}\n")


    def generate_prompt(self, input_text, cot=None, previous_answer=None, feedback=None, include_scoring=True):
        prompt = ""

        if cot:
            cot_filled = cot.replace("{replace with the prompt}", input_text.strip())
            prompt += cot_filled + "\n\n"
        else:
            prompt += f"Q: {input_text.strip()}\n"

        if previous_answer:
            prompt += f"Previous Answer: {previous_answer.strip()}\n"

        if previous_answer:
            prompt += f"Previous Answer: {previous_answer}\n"
        if feedback:
            prompt += f"Feedback: {feedback}\n"
        prompt += "Based on the feedback above, please provide a revised and improved answer:\nA: "

        return prompt


    def generating_answer(self, prompt, cot=None, previous_answer=None, feedback=None, max_new_tokens=200):
        full_prompt = self.generate_prompt(prompt, cot, previous_answer, feedback)

        print(f"\nprompt:\n{full_prompt}\n")

        inputs = self.tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=1024)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        output_ids = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.5,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )

        output_text = self.tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

        if output_text.startswith(full_prompt):
            output_text = output_text[len(full_prompt):].strip()

        print(f"Answer: {output_text}")

        pred_score, feedback = self.feedback_model.predict_score_and_feedback(prompt, output_text)
        return output_text, pred_score, feedback

    def check_feedback_satisfaction(self, prompt, previous_answer, previous_feedback, current_answer):
        """
        Check if the current answer improves upon the previous answer based on the feedback.
        If not, return what needs further improvement.
        """
        review_prompt = (
            f"Question: {prompt}\n\n"
            f"Previous Answer: {previous_answer}\n\n"
            f"Feedback: {previous_feedback}\n\n"
            f"Current Answer: {current_answer}\n\n"
            "Does the current answer successfully address the feedback provided earlier? "
            "Reply with 'Yes' or 'No'. If 'No', explain which part still needs improvement."
        )

        try:
            response = openai.chat.completions.create(
                model=self.feedback_model.model_name,
                messages=[
                    {"role": "system", "content": "You are a critical reviewer that checks whether the current answer improves based on previous feedback."},
                    {"role": "user", "content": review_prompt}
                ],
                temperature=0.5,
                max_tokens=150,
            )

            output = response.choices[0].message.content.strip()
            print(f"Feedback Satisfaction Check:\n{output}\n")
            return output.lower().startswith("yes"), output

        except Exception as e:
            print(f"OpenAI API error in check_feedback_satisfaction: {e}")
            return False, "API Error"

    def update_model_with_feedback(self, prompt, previous_answer, previous_feedback, cot=None):
        self.model.train()

        if isinstance(prompt, str):
            prompts = [prompt]
        else:
            prompts = prompt

        full_prompt = self.generate_prompt(
            input_text=prompts[0],
            cot=cot,
            previous_answer=previous_answer,
            feedback=previous_feedback,
            include_scoring=True
        )

        inputs = self.tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=512).to(self.device)

        generated_ids = self.model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=2,
            do_sample=True,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )

        improved_answer = self.tokenizer.decode(generated_ids[0], skip_special_tokens=True)[len(full_prompt):].strip()

        print(f"Feedback: {previous_feedback}")

        prev_score, _ = self.feedback_model.predict_score_and_feedback(prompts[0], previous_answer)
        improved_score, _ = self.feedback_model.predict_score_and_feedback(prompts[0], improved_answer)
        delta = improved_score - prev_score

        combined_text = full_prompt + improved_answer
        encodings = self.tokenizer(combined_text, return_tensors="pt", truncation=True, max_length=512).to(self.device)
        input_ids = encodings.input_ids
        attention_mask = encodings.attention_mask

        prompt_len = len(self.tokenizer(full_prompt, truncation=True, max_length=512).input_ids)
        labels = input_ids.clone()
        labels[:, :prompt_len] = -100

        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        if delta >= 0:
            reward = max(delta, 0.1)
        else:
            reward = delta * self.penalty_factor

        min_reward_magnitude = 0.05
        if abs(reward) < min_reward_magnitude:
            reward = reward / abs(reward) * min_reward_magnitude

        is_satisfied, critique = self.check_feedback_satisfaction(prompt, previous_answer, previous_feedback, improved_answer)
        if is_satisfied:
            reward += 0.2
        else:
            reward -= 0.2

        policy_loss = -reward * loss

        self.optimizer.zero_grad()
        policy_loss.backward()
        self.optimizer.step()

        return improved_answer, critique, improved_score, reward

    def self_critique(self, prompt, previous_answer, previous_feedback, cot=None, iterations=10, target_score=0.95):
        best_answer = previous_answer
        best_score, _ = self.feedback_model.predict_score_and_feedback(prompt, previous_answer)
        feedback = previous_feedback

        previous_answers = set()
        previous_answers.add(best_answer)
        for step in range(iterations):
            if best_score >= target_score:
                print(f"Target score reached ({best_score:.4f}), stopping early.")
                break

            improved_answer, feedback, improved_score, reward = self.update_model_with_feedback(
                prompt, best_answer, feedback, cot
            )

            if improved_answer in previous_answers:
                reward -= 0.1
                print("Penalty: Repeated answer detected.")

            previous_answers.add(improved_answer)

            print(f"Step {step}: Reward {reward:.4f}, Score {improved_score:.4f}")
            print(f"Improved Answer:\n{improved_answer}\n")

            if improved_score > best_score:
                best_answer = improved_answer
                best_score = improved_score


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_name = "gpt2-medium"

base_tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
penalty_factor = 2
base = BaseModel_Test(base_model, base_tokenizer, gpt_model, penalty_factor)

In [ ]:
base.fine_tuning()

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.



Batch 0 Prediction: Q: when did the first rick and morty come out
Previous Answer: Rick and Morty Rick and Morty is an American adult animated science-fiction sitcom created by Justin Roiland and Dan Harmon for Cartoon Network's late-night programming block Adult Swim. The series follows the misadventures of cynical mad scientist Rick Sanchez and his fretful, easily influenced grandson Morty Smith, who split their time between domestic life and interdimensional adventures. The series premiered on December 2, 2013, and the third season concluded on October 1, 2017. A fourth season has been mentioned, first by Harmon in a September 2017 interview, and later in the post-credits scene of the third season's finale.
Feedback: The answer is accurate, complete, and well-written. It not only provides the exact date when the first Rick and Morty episode came out, but also gives additional information about the creators, the premise of the show, and its subsequent seasons.
Based on the feedback 

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.



Batch 10 Prediction: Q: what is the android version of redmi note 4
Previous Answer: Xiaomi Redmi Note 4 Xiaomi Redmi Note 4- codename nikel is equipped with MediaTek MT6796 Helio X20 CPU and Mali-T880 MP4 GPU. It runs MIUI 8, updated to MIUI 9 based on Android 6.0 . The 4X - codename mido variants have the Octa-core 2.0Â GHz Snapdragon 625 CPU and Adreno 506 GPU.[10] It runs MIUI 8 based on Android 6.0, which was updated to MIUI 9 based on Android 7.0.
Feedback: The answer is accurate and complete. It provides detailed information about the Android version of Redmi Note 4, including the updates it received. The answer also includes additional information about the device's CPU and GPU, which, while not directly asked for, adds to the completeness of the response.
Based on the feedback above, please provide a revised and improved answer:
A: The Xiaomi Redmi Note 4 originally comes with Android Marshmallow (6.0) under the MIUI 8 user interface. There have been updates to this system, a

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


...


Validation Batch 46 Prediction: Q: where was where the red fern grows filmed
Previous Answer: Where the Red Fern Grows (1974 film) The film was produced by Lyman Dayton and filmed on location in Oklahoma.[4][5]
Feedback: The answer is accurate, complete, and well-written. It correctly identifies the filming location of "Where the Red Fern Grows" as Oklahoma.
Based on the feedback above, please provide a revised and improved answer:
A: "Where the Red Fern Grows," the 1974 film adaptation of the beloved novel, was shot on location in the scenic landscapes of Oklahoma, under the production of Lyman Dayton. This provided an authentic backdrop for the story.The movie has been widely recognized throughout the world as one that explores themes ranging from environmentalism to social justice, providing viewers with fascinating insights into contemporary issues such love and family relationships at their core; these portrayals have become iconic within popular culture over time due both critic

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.



Validation Batch 47 Prediction: Q: where is the syrian desert located on a map
Previous Answer: Syrian Desert The Syrian Desert (Arabic: بادية الشام‎, Bâdiyat aş-Şâm), also known as the Hamad,[1] is a combination of steppe and desert covering 500,000 square kilometers (200,000 square miles) of the Middle East, including parts of south-eastern Syria, northeastern Jordan, northern Saudi Arabia, and western Iraq. To the south it borders and merges into the Arabian Desert.[2] The land is open, gravely desert pavement, cut with occasional wadis.[3][4]
Feedback: The answer is accurate, complete, and well-written. It provides a detailed description of the location of the Syrian Desert, including the countries it spans and its geographical features.
Based on the feedback above, please provide a revised and improved answer:
A: The Syrian Desert, also known as the Hamad, is a vast expanse of approximately 500,000 square kilometers that spans across multiple countries in the Middle East. It is l

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.



Validation Batch 48 Prediction: Q: can you buy beer on sunday in pa
Previous Answer: Alcohol laws of Pennsylvania Pennsylvania is an alcoholic beverage control state. Spirits are to be sold only in the state owned Fine Wine and Good Spirits stores, which also sell wine, but not beer. Prices are generally the same throughout the state, but state stores may offer special discounts and sales,[4] and county sales tax may cause the price to differ slightly. People under the age of 21 are allowed to enter Fine Wine and Good Spirits stores, contrary to popular belief, but only if accompanied by a parent or guardian. Monday through Saturday, a store may open as early as 9 am and close as late as 10 pm. On Sunday, many stores sell liquor from 11 am until 7 pm.[5]
Feedback: The answer provides a lot of information about the alcohol laws in Pennsylvania, but it does not directly answer the question about whether or not you can buy beer on Sunday. The information about the sale of spirits and win

In [ ]:
answer_few_shot, score_few_shot, feedback_few_shot = base.generating_answer(prompt, cot)


prompt:
Q: California is nice.
A: California is renowned for its diverse geography and pleasant climate. From the sun-drenched beaches to the towering redwoods, the state offers incredible natural beauty. Its vibrant cities like Los Angeles and San Francisco attract millions with their culture and opportunities.

Q: What are the benefits of being a machine learning engineer?

Based on the feedback above, please provide a revised and improved answer:
A: 

Answer: I can learn from my mistakes in order not make them again A: I have learned many things through experience B :   My job has taught me that there's no one way or right answers C(and more importantly) D: Not all solutions will be perfect but if you find something wrong then it means we're trying too hard here so let us try harder E (you know what?) F: If your solution doesn't work out because someone else did better go back into coding G/H: Have fun!


In [ ]:
print(score_few_shot)

-0.5


In [ ]:
print(feedback_few_shot)

The response lacks clarity and focus on the specific benefits of being a machine learning engineer. It mentions learning from mistakes but does not provide concrete benefits such as career opportunities, salary, or impact on technology. Overall, it is off-topic and unhelpful.


In [ ]:
base.self_critique(prompt, answer_few_shot, feedback_few_shot, iterations = 10)

Feedback: The response lacks clarity and focus on the specific benefits of being a machine learning engineer. It mentions learning from mistakes but does not provide concrete benefits such as career opportunities, salary, or impact on technology. Overall, it is off-topic and unhelpful.


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Feedback Satisfaction Check:
No. The current answer does not address the feedback provided earlier. It still lacks clarity and does not focus on the specific benefits of being a machine learning engineer, such as career opportunities, salary, or impact on technology. Instead, it presents a series of disjointed thoughts that do not contribute meaningfully to the question. The response needs to be more structured and should directly highlight the advantages of the profession.

Step 0: Reward -1.2000, Score -1.0000
Improved Answer:
I think both to read about this problem well before starting any project should allow for an understanding of how our algorithms operate J; And don`t get involved with people who seem like they might need help when really just want to talk at some point during their day H: Is still very much important though . You could also ask "how do i improve" K = Make sure everyone knows why he works together instead j= Keep his own team happy M&n> Try lots other ways besi

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
save_path = "/content/drive/MyDrive/finetuned_base_model"

base.model.save_pretrained(save_path)
base.tokenizer.save_pretrained(save_path)

NameError: name 'base' is not defined

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

save_path = "/content/drive/MyDrive/finetuned_base_model"

base_tokenizer = AutoTokenizer.from_pretrained(save_path)
base_model = AutoModelForCausalLM.from_pretrained(save_path)

## Amateur Feedback Model

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch.nn as nn
import torch

class AmateurFeedbackModel(torch.nn.Module):
    def __init__(self, model_name="gpt2"):
        super().__init__()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = GPT2Tokenizer.from_pretrained(model_name)
        self.model = GPT2LMHeadModel.from_pretrained(model_name)
        self.score_head = nn.Sequential(
            nn.Linear(self.model.config.n_embd, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Tanh()
        )

        self.to(self.device)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            output_hidden_states=True,
            return_dict=True,
        )
        hidden_state = outputs.hidden_states[-1][:, 0, :]
        score = self.score_head(hidden_state).squeeze(-1)
        return outputs.loss, outputs.hidden_states, score

    def predict_score_and_feedback(self, prompt_text, answer):
        prompt = (
            "You are an expert evaluator. Your task is to score the quality of the following answer based on how well it responds to the given prompt.\n"
            "Evaluate the answer on a scale from -1 to 1, where:\n"
            "-1 = completely incorrect, irrelevant, or does not address the prompt\n"
            "0 = neutral or partially addresses the prompt\n"
            "1 = excellent, fully correct, detailed, and thoroughly addresses the prompt\n\n"
            f"Prompt: {prompt_text}\n"
            f"Answer: {answer}\n\n"
            "Respond exactly in this format:\n"
            "Score: <score between -1 and 1>\n"
            "Feedback: <short, constructive feedback on how to improve the answer>\n"
        )

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)

        with torch.no_grad():
            output = self.model.generate(
                **inputs,
                max_new_tokens=400,
                do_sample=True,
                top_k=50,
                top_p=0.95,
                temperature=0.8,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        decoded = self.tokenizer.decode(output[0], skip_special_tokens=True)
        generated = decoded[len(prompt):].strip()

        try:
            lines = generated.splitlines()
            score_line = next(line for line in lines if line.lower().startswith("score:"))
            feedback_line = next(line for line in lines if line.lower().startswith("feedback:"))
            score = float(score_line.split(":", 1)[1].strip())
            feedback = feedback_line.split(":", 1)[1].strip()
        except Exception:
            score = 0.0
            feedback = f"Failed to parse model output: {generated}"

        return score, feedback

In [ ]:
amateur_feedback_model = AmateurFeedbackModel()

## Knowledge Distillation Network

In [ ]:
!pip install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 107.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [ ]:
import json

file_path = "1k_dataset.jsonl"
expert_datasets = []

with open(file_path, "r") as f:
    for line in f:
        item = json.loads(line)

        prompt = item["prompt"]
        original_answer = item["original_answer"]
        score = item["score"]
        feedback = item["feedback"]
        expert_datasets.append(
            {"prompt": prompt,
             "answer": original_answer,
             "feedback": feedback,
             "score": score}
        )

print(f" {len(expert_datasets)} examples loaded")

 1000 examples loaded


In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from bert_score import score as bert_score


class AmateurExpertFeedbackNetWork:
    def __init__(self, student, teacher, expert_datasets=None, device=None, optimizer=None):
        self.student = student
        self.teacher = teacher
        self.expert_datasets = expert_datasets or []
        self.student_tokenizer = student.tokenizer
        self.teacher_tokenizer = teacher.tokenizer
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")

        if self.student_tokenizer.pad_token is None:
            self.student_tokenizer.pad_token = self.student_tokenizer.eos_token

        self.optimizer = optimizer if optimizer else AdamW(self.student.model.parameters(), lr=5e-5)
        self.align_hidden = None  # for hidden state matching

    def knowledge_distillation_using_SFT(self, epochs=3, stopped=None):
        score_loss_fn = nn.MSELoss()

        for epoch in range(epochs):
            print(f"\n--- Epoch {epoch+1}/{epochs} ---")
            total_loss_accum = 0

            for i, expert_sample in enumerate(self.expert_datasets):
                if stopped is not None and i == stopped:
                    break

                prompt_text = expert_sample.get("prompt")
                answer = expert_sample.get("answer")
                expert_score = expert_sample.get("score")
                expert_feedback = expert_sample.get("feedback")

                if not prompt_text or not answer or expert_score is None:
                    continue

                # Prediction before fine-tuning (for tracking)
                student_score, student_feedback = self.student.predict_score_and_feedback(prompt_text, answer)
                print(f"\nSample {i+1}")
                print(f"Student Score (Before): {student_score:.2f}")
                print(f"Expert Score          : {expert_score:.2f}")

                # Prompt construction
                full_prompt = (
                    "You are an expert evaluator. Your task is to score the quality of the following answer based on how well it responds to the given prompt.\n"
                    "Evaluate the answer on a scale from -1 to 1, where:\n"
                    "-1 = completely incorrect, irrelevant, or does not address the prompt\n"
                    "0 = neutral or partially addresses the prompt\n"
                    "1 = excellent, fully correct, detailed, and thoroughly addresses the prompt\n\n"
                    f"Prompt: {prompt_text}\n"
                    f"Answer: {answer}\n\n"
                    "Respond exactly in this format:\n"
                    "Score: <score between -1 and 1>\n"
                    "Feedback: <short, constructive feedback on how to improve the answer>\n"
                )

                # Tokenize input and labels
                student_inputs = self.student_tokenizer(full_prompt, return_tensors="pt", truncation=True, padding=True).to(self.device)
                labels = self.student_tokenizer(expert_feedback, return_tensors="pt", truncation=True, padding=True).input_ids.to(self.device)

                # Pad labels if needed
                if labels.size(1) < student_inputs["input_ids"].size(1):
                    pad_len = student_inputs["input_ids"].size(1) - labels.size(1)
                    padding = torch.full((labels.size(0), pad_len), -100, dtype=torch.long, device=self.device)
                    labels = torch.cat([labels, padding], dim=1)

                # Student forward pass (with hidden states)
                student_out = self.student.model(
                    input_ids=student_inputs["input_ids"],
                    attention_mask=student_inputs["attention_mask"],
                    labels=labels,
                    output_hidden_states=True,
                    return_dict=True
                )
                lm_loss = student_out.loss
                student_hidden = student_out.hidden_states[-1][:, 0, :]  # CLS

                # Teacher forward pass (no grad)
                with torch.no_grad():
                    teacher_inputs = self.teacher_tokenizer(full_prompt, return_tensors="pt", truncation=True, padding=True).to(self.device)
                    teacher_out = self.teacher.model(
                        input_ids=teacher_inputs["input_ids"],
                        attention_mask=teacher_inputs["attention_mask"],
                        output_hidden_states=True,
                        return_dict=True
                    )
                    teacher_hidden = teacher_out.hidden_states[-1][:, 0, :]

                # Hidden layer alignment loss
                if self.align_hidden is None:
                    student_dim = student_hidden.size(-1)
                    teacher_dim = teacher_hidden.size(-1)
                    self.align_hidden = (
                        nn.Linear(student_dim, teacher_dim).to(self.device)
                        if student_dim != teacher_dim else nn.Identity()
                    )

                student_aligned = self.align_hidden(student_hidden)
                cos_sim = nn.functional.cosine_similarity(student_aligned, teacher_hidden, dim=-1)
                hidden_loss = 1.0 - cos_sim.mean()

                # Score regression loss
                student_score_tensor = torch.tensor([student_score], dtype=torch.float32, device=self.device)
                expert_score_tensor = torch.tensor([expert_score], dtype=torch.float32, device=self.device)
                score_loss = score_loss_fn(student_score_tensor, expert_score_tensor)

                # Semantic alignment (BERTScore)
                P, R, F1 = bert_score([student_feedback], [expert_feedback], lang="en", verbose=False)
                semantic_loss = 1.0 - F1[0].item()
                semantic_loss_tensor = torch.tensor(semantic_loss, requires_grad=True, device=self.device)

                # Total loss (no logits KL)
                alpha, beta, gamma, delta = 1.0, 0.5, 0.5, 0.5
                total_loss = (
                    alpha * lm_loss +
                    beta * score_loss +
                    gamma * semantic_loss_tensor +
                    delta * hidden_loss
                )

                # Optimize
                self.optimizer.zero_grad()
                total_loss.backward()
                self.optimizer.step()

                # Logging
                print(f"LM Loss          : {lm_loss.item():.4f}")
                print(f"Score Loss       : {score_loss.item():.4f}")
                print(f"Semantic Loss    : {semantic_loss_tensor.item():.4f}")
                print(f"Hidden Align Loss: {hidden_loss.item():.4f}")
                print(f"Total Loss       : {total_loss.item():.4f}")

                total_loss_accum += total_loss.item()

            avg_loss = total_loss_accum / len(self.expert_datasets) if self.expert_datasets else 0.0
            print(f"\nEpoch {epoch+1} average loss: {avg_loss:.4f}")

        return self.student, self.teacher


In [ ]:
network = AmateurExpertFeedbackNetWork(amateur_feedback_model, expert, expert_datasets)

In [ ]:
network.knowledge_distillation_using_SFT()

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Starting fine-tune on 1000 samples

--- Epoch 1/3 ---

Sample 1
Student Score (Before): 0.00
Expert Score          : 1.00
Teacher Feedback Only:
Score: None
Feedback: You are an expert evaluator. Your task is to score the quality of the following answer based on how well it responds to the given prompt.

Use the following detailed scoring scale:
1.0: Perfect completion with no issues.
0.5 – 0.9: Generally good, minor flaws.
0.1 – 0.4: Partially helpful, moderate issues.
0.0: Neutral or off-topic, neither helpful nor harmful.
-0.1 – -0.4: Mildly harmful or misleading.
-0.5 – -0.9: Significantly off-task or misleading.
-1.0: Complete failure or harmful content.

Provide your reasoning and feedback in no more than 50 words, focusing on what was done well and what was missed.
Do not comment on the length or verbosity of the answer.

Respond exactly in this format:
[reason]xxxx (MAX 50 words). Score: [score]

Prompt: who are the basques and where do they live
Answer: Basques The Basques (/b

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LM Loss          : 1.5707
Score Loss       : 1.0000
Semantic Loss    : 0.2048
Hidden Align Loss: 1.0238
Total Loss       : 2.6850


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LM Loss          : 0.5369
Score Loss       : 0.2500
Semantic Loss    : 0.1747
Hidden Align Loss: 1.0217
Total Loss       : 1.2601

Sample 4
Student Score (Before): 0.00
Expert Score          : 1.00
Teacher Feedback Only:
Score: None
Feedback: You are an expert evaluator. Your task is to score the quality of the following answer based on how well it responds to the given prompt.

Use the following detailed scoring scale:
1.0: Perfect completion with no issues.
0.5 – 0.9: Generally good, minor flaws.
0.1 – 0.4: Partially helpful, moderate issues.
0.0: Neutral or off-topic, neither helpful nor harmful.
-0.1 – -0.4: Mildly harmful or misleading.
-0.5 – -0.9: Significantly off-task or misleading.
-1.0: Complete failure or harmful content.

Provide your reasoning and feedback in no more than 50 words, focusing on what was done well and what was missed.
Do not comment on the length or verbosity of the answer.

Respond exactly in this format:
[reason]xxxx (MAX 50 words). Score: [score]

Prompt

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LM Loss          : 1.3340
Score Loss       : 1.0000
Semantic Loss    : 0.1672
Hidden Align Loss: 1.0213
Total Loss       : 2.4283

Sample 5
Student Score (Before): 0.00
Expert Score          : 0.50


KeyboardInterrupt: 

In [ ]:
import os
os.makedirs('expert_model_checkpoint', exist_ok=True)

**Build Dataset**

In [ ]:
def build_kd_dataset(results):
    kd_data = []
    for ex in results:
        inp = f"Question: {ex['input']}\nStudent Answer: {ex['x0']}"
        out = ex['p_expert']
        kd_data.append({"input": inp, "output": out})
    return kd_data

In [ ]:
#Saving FILE

import json
with open("kd_data.json", "w") as f:
    json.dump(build_kd_dataset(results), f, indent=2)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

model = AutoModelForCausalLM.from_pretrained("gpt2-medium")
tokenizer = AutoTokenizer.from_pretrained("gpt2-medium")

def tokenize(example):
    prompt = example["input"]
    target = example["output"]
    tokenized_input = tokenizer(prompt, truncation=True, padding="max_length", max_length=512)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(target, truncation=True, padding="max_length", max_length=128)
    tokenized_input["labels"] = labels["input_ids"]
    return tokenized_input

from datasets import load_dataset
dataset = load_dataset("json", data_files="kd_data.json")["train"]
dataset = dataset.map(tokenize)

args = TrainingArguments(
    output_dir="./student-kd-checkpoints",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    learning_rate=5e-5,
    fp16=True,
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    tokenizer=tokenizer
)

trainer.train()


In [ ]:
from transformers import pipeline
student_model = AutoModelForCausalLM.from_pretrained("./student-kd-checkpoints")
student_pipe = pipeline("text-generation", model=student_model, tokenizer=tokenizer)

# Run feedback generation or refinement with updated student

## Experiments CODE
We haven't integrated with the real models that we'll use YET. Crucially some of that code has been written but commented out for the time being.

**Fine Tuning Student Model Using SFT**

**Evaluate Distilled Student**

Imports and Stuff

In [ ]:
!pip install transformers
!pip uninstall datasets
!pip install datasets
!pip install transformers datasets accelerate sentence-transformers --quiet
#!pip install openai --quiet

Hugging Face Login

In [ ]:
!huggingface-cli login

Experiment Pipeline

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from datasets import load_dataset
from bert_score import score as bert_score
from transformers import AutoTokenizer
from nltk.translate.bleu_score import sentence_bleu
from rouge import Rouge

from expert_model import ExpertClass
from amateur_model import AmateurFeedbackModel

# ⚙️ Load pre-trained models (Base = Teacher, Amateur = Post-KD Student)
base_model = ExpertClass(model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0", model_path="base_model")
amateur_model = AmateurFeedbackModel(model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0", model_path="amateur_model")

# 📊 Initialize containers for metrics
results = {
    "dataset": [],
    "model": [],
    "accuracy": [],
    "bleu": [],
    "rouge": [],
    "bertscore": [],
    "toxicity": [],
    "avg_output_length": [],
}

# 📚 Dataset-specific evaluation logic
def evaluate_dataset(dataset_name, model):
    tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0", use_fast=True)
    predictions, references = [], []
    total_correct = 0
    total_samples = 0
    all_lengths = []

    # Load dataset
    if dataset_name == "gsm8k":
        data = load_dataset("gsm8k", "main", split="test")
        for sample in tqdm(data):
            prompt = sample["question"]
            gt_answer = sample["answer"]
            output = model.generate_feedback(prompt, "")
            predictions.append(output)
            references.append(gt_answer)
            if gt_answer.strip() in output:
                total_correct += 1
            all_lengths.append(len(output.split()))
            total_samples += 1

    elif dataset_name == "commongen":
        data = load_dataset("common_gen", split="test[:100]")
        for sample in tqdm(data):
            concept_string = ', '.join(sample["concepts"])
            prompt = f"Write a coherent sentence using the concepts: {concept_string}"
            gt = sample["target"][0]
            output = model.generate_feedback(prompt, "")
            predictions.append(output)
            references.append(gt)
            all_lengths.append(len(output.split()))
            total_samples += 1

    elif dataset_name == "whats_that_book":
        data = load_dataset("GEM/whats_that_book", split="test[:100]")
        for sample in tqdm(data):
            prompt = f"Q: {sample['question']}"
            gt = sample["answer"]
            output = model.generate_feedback(prompt, "")
            predictions.append(output)
            references.append(gt)
            all_lengths.append(len(output.split()))
            total_samples += 1

    elif dataset_name == "real_toxicity_prompts":
        data = load_dataset("allenai/real-toxicity-prompts", split="train[:100]")
        for sample in tqdm(data):
            prompt = sample["prompt"]["text"]
            output = model.generate_feedback(prompt, "")
            predictions.append(output)
            references.append("")
            all_lengths.append(len(output.split()))
            total_samples += 1

    elif dataset_name == "agieval":
        data = load_dataset("agieval", "aqua_rat", split="test[:100]")
        for sample in tqdm(data):
            prompt = sample["question"]
            gt = sample["answer"]
            output = model.generate_feedback(prompt, "")
            predictions.append(output)
            references.append(gt)
            if gt.strip() in output:
                total_correct += 1
            all_lengths.append(len(output.split()))
            total_samples += 1

    # METRICS
    _, _, bert_f1 = bert_score(predictions, references, lang="en", verbose=False)
    rouge = Rouge()
    rouge_scores = rouge.get_scores(predictions, references, avg=True)
    bleu_scores = [sentence_bleu([ref.split()], pred.split()) for ref, pred in zip(references, predictions)]

    avg_bleu = np.mean(bleu_scores)
    avg_rouge = rouge_scores["rouge-l"]["f"]
    avg_bertscore = bert_f1.mean().item()
    avg_length = np.mean(all_lengths)
    acc = total_correct / total_samples if "gsm8k" in dataset_name or "agieval" in dataset_name else None

    return acc, avg_bleu, avg_rouge, avg_bertscore, avg_length


Experiment Part

In [ ]:
datasets = [
    "gsm8k",
    "commongen",
    "whats_that_book",
    "real_toxicity_prompts",
    "agieval"
]

models = {
    "Base Model (Teacher)": base_model,
    "Amateur Model (Post-KD)": amateur_model
}

for dataset in datasets_to_test:
    for model_name, model_instance in models.items():
        acc, bleu, rouge, bert_f1, avg_len = evaluate_dataset(dataset, model_instance)
        results["dataset"].append(dataset)
        results["model"].append(model_name)
        results["accuracy"].append(acc)
        results["bleu"].append(bleu)
        results["rouge"].append(rouge)
        results["bertscore"].append(bert_f1)
        results["avg_output_length"].append(avg_len)


Visualizations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Replace with your actual evaluation results
results = {
    "dataset": [
        "gsm8k", "gsm8k",
        "agieval", "agieval"
    ],
    "model": [
        "Base Model (Teacher)", "Amateur Model (Post-KD)",
        "Base Model (Teacher)", "Amateur Model (Post-KD)"
    ],
    "accuracy": [
        0.62, 0.68,
        0.55, 0.63
    ],
    "bleu": [None] * 4,
    "rouge": [None] * 4,
    "bertscore": [None] * 4,
    "avg_output_length": [None] * 4
}

df = pd.DataFrame(results)

# Bar plot
fig, ax = plt.subplots(figsize=(10, 6))
datasets = ["gsm8k", "agieval"]
x = np.arange(len(datasets))
bar_width = 0.35

teacher_acc = df[df["model"] == "Base Model (Teacher)"]["accuracy"].values
student_acc = df[df["model"] == "Amateur Model (Post-KD)"]["accuracy"].values

ax.bar(x - bar_width/2, teacher_acc, width=bar_width, label='Base Model (Teacher)', color='steelblue')
ax.bar(x + bar_width/2, student_acc, width=bar_width, label='Amateur Model (Post-KD)', color='orange')

ax.set_xlabel("Dataset")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy Comparison Between Teacher and Amateur Models")
ax.set_xticks(x)
ax.set_xticklabels(datasets)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


## Archived the old Code